In [ ]:
# ann_train_pfq.py

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import save_model

In [ ]:
# Load data
df = pd.read_csv('pfq.csv')

In [ ]:
#  Separate features and targets
X = df.drop(columns=['Fertilizer_Used(tons)', 'Pesticide_Used(kg)'])
y_fert = df['Fertilizer_Used(tons)']
y_pest = df['Pesticide_Used(kg)']

In [ ]:
#Label encode categorical features
categorical_cols = ['Crop_Type', 'Season', 'Soil_Type', 'Irrigation_Type']
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

# Save column order for future prediction
feature_columns = X.columns.tolist()

# 

In [ ]:
 # Train/test split
X_train, X_test, y_fert_train, y_fert_test, y_pest_train, y_pest_test = train_test_split(
    X, y_fert, y_pest, test_size=0.2, random_state=42
)

In [ ]:
# ANN model for Fertilizer
fert_model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])
fert_model.compile(optimizer='adam', loss='mean_squared_error')

fert_model.fit(X_train, y_fert_train, epochs=100, batch_size=16, verbose=0)

In [ ]:
#  ANN model for Pesticide
pest_model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])
pest_model.compile(optimizer='adam', loss='mean_squared_error')

# pest_model.compile(optimizer='adam', loss='mse')
pest_model.fit(X_train, y_pest_train, epochs=100, batch_size=16, verbose=0)

In [ ]:
# Save models and encoders
fert_model.save('fertilizer_model_ann.keras')
pest_model.save('pesticide_model_ann.keras')
joblib.dump(encoders, 'pfq_feature_encoders.pkl')
joblib.dump(feature_columns, 'pfq_feature_columns.pkl')

C:\Users\Manish Kumar Jain\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Manish Kumar Jain\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


['pfq_feature_columns.pkl']

In [12]:
# ann_predict_pfq.py

import pandas as pd
import joblib
import numpy as np
from tensorflow.keras.models import load_model

# 1. Load models and encoders once
fertilizer_model = load_model('fertilizer_model_ann.keras')
pesticide_model = load_model('pesticide_model_ann.keras')
encoders = joblib.load('pfq_feature_encoders.pkl')
feature_columns = joblib.load('pfq_feature_columns.pkl')

# 2. Define a single prediction function outside loops
def prepare_sample(input_dict):
    sample = pd.DataFrame([input_dict])
    

    for col, le in encoders.items():
        sample[col] = le.transform(sample[col])


    for col in feature_columns:
        if col not in sample.columns:
            sample[col] = 0
    sample = sample[feature_columns]
    
    return sample.to_numpy()

# 3. Define your input
input_data = {
    'Crop_Type': 'wheat',
    'Season': 'rabi',
    'Soil_Type': 'loamy',
    'Temperature': 29999990,
    'Humidity': 78,
    'Rainfall': 120,
    'PH': 6.5,
    'Yield(Tons)': 5005,
    'Irrigation_Type': 'drip'
}

# 4. Prepare input and predict
input_array = prepare_sample(input_data)

fertilizer_pred = fertilizer_model.predict(input_array, verbose=0)[0][0]
pesticide_pred = pesticide_model.predict(input_array, verbose=0)[0][0]

# 5. Output
print(f"✅ Predicted Fertilizer Usage (tons): {fertilizer_pred:.2f}")
print(f"✅ Predicted Pesticide Usage (kg): {pesticide_pred:.2f}")


✅ Predicted Fertilizer Usage (tons): 0.23
✅ Predicted Pesticide Usage (kg): 1.55
